# 03 — Fetch Missing Logos
Fills `Organization Logo White` in Airtable for any partner record where that field is currently empty.

**Run order:** cells 1–9.7 fetch + validate + store locally. Cell 10 is the write-back to Airtable — run it only after you are satisfied with the downloads.

Logo sources (in order):
- **Source A:** Clearbit Logo API — `https://logo.clearbit.com/{domain}`
- **Source B:** Google S2 favicon — `https://www.google.com/s2/favicons?domain={domain}&sz=256`

In [23]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import os
import re
import time
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from dotenv import load_dotenv
from pyairtable import Api

load_dotenv()
print("imports OK")

imports OK


In [24]:
# ── Cell 2 · Constants ────────────────────────────────────────────────────────
AIRTABLE_BASE_ID   = "appIYFN5sAJzK1bPg"
PARTNER_TABLE_ID   = "tbl2FMZOARI7I66fq"
LOGO_FIELD_NAME    = "Organization Logo White"   # exact Airtable column name
LOGOS_DIR          = Path("public") / "logos"

CLEARBIT_TMPL      = "https://logo.clearbit.com/{domain}"
GOOGLE_S2_TMPL     = "https://www.google.com/s2/favicons?domain={domain}&sz=256"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; logo-fetcher/1.0)"
}
TIMEOUT = 15   # seconds per request

LOGOS_DIR.mkdir(parents=True, exist_ok=True)
print(f"logos dir: {LOGOS_DIR.resolve()}")

logos dir: /Users/harshitachakravadhanula./Desktop/UN/Craf'd work/code/parters_interactive/crafd-donor-portal/public/logos


In [25]:
# ── Cell 3 · Load data from Airtable (keep record IDs for write-back) ─────────
api   = Api(os.environ["AIRTABLE_API_KEY"])
table = api.table(AIRTABLE_BASE_ID, PARTNER_TABLE_ID)

# Fetch with string cell format so attachment fields come back as
# "filename.ext (https://url)" strings — same as the existing scripts.
raw_records = table.all(
    cell_format="string",
    user_locale="en-ca",
    time_zone="America/New_York",
)

# Build a DataFrame that keeps the Airtable record ID alongside the fields.
rows = []
for rec in raw_records:
    row = {"_record_id": rec["id"]}
    row.update(rec["fields"])
    rows.append(row)

df_raw = pd.DataFrame(rows)
print(f"fetched {len(df_raw)} records")
df_raw.columns.tolist()

fetched 182 records


['_record_id',
 'Organization name',
 "CRAF'd partner type",
 'Short name',
 'Is CRAFd Project',
 'Organization Type',
 'UN-Organization',
 'Website',
 'Source',
 'Ecosystem Stakeholder Type',
 'Total Project Grant Size',
 "Support for CRAF'd projects",
 'Operating Country',
 '[?] Type of organization',
 'Signed MoU Or Framework Agreement?',
 'Contacts',
 '[Ecosystem Mapping] Organization Logo White PNG/SVG',
 'Comments/Notes',
 'Women-Led/Feminist',
 'Global South?',
 'Year MoU/ Agreement Signed',
 'MOU/ HACT signing date',
 'Organization – Department',
 'HACT Assessment Date',
 'Exact Grant Size',
 'Job Posting Website',
 'Projects (Lead)',
 'Projects',
 'Outgoing 3',
 'Signed PSEA assessment',
 'Matched Compass Orgs',
 'org_key (from Matched Compass Orgs)',
 'Org Type (from Matched Compass Orgs)',
 'Organization Logo Color',
 'Received Grants ',
 'Outgoing 2',
 'org_id']

In [26]:
# ── Cell 4 · Rename & select columns ─────────────────────────────────────────
rename_mapping = {
    "Organization name":     "org_full_name",
    "Short name":            "org_short_name",   # note: exact capitalisation from API
    "Website":               "website",
    "Organization Logo White": "org_logo_white",
    "org_id":                "org_id",
}

df = df_raw.rename(columns=rename_mapping)

# Keep only the columns we need (plus the internal record id).
keep = ["_record_id", "org_short_name", "org_full_name", "website", "org_logo_white"]
keep = [c for c in keep if c in df.columns]   # gracefully skip any missing
df = df[keep].copy()

df["org_logo_white"] = df["org_logo_white"].fillna("")
df["website"]        = df["website"].fillna("")
df["org_short_name"] = df["org_short_name"].fillna("")

print(df[["org_short_name", "website", "org_logo_white"]].head(10).to_string())

KeyError: 'org_logo_white'

In [ ]:
# ── Cell 5 · Filter: only records with no logo ────────────────────────────────
df_missing = df[df["org_logo_white"].str.strip() == ""].copy()
df_missing = df_missing.reset_index(drop=True)

total     = len(df)
n_filled  = total - len(df_missing)
n_missing = len(df_missing)

print(f"total records  : {total}")
print(f"already filled : {n_filled}")
print(f"missing logos  : {n_missing}")
print()
df_missing[["org_short_name", "org_full_name", "website"]]

In [ ]:
# ── Cell 6 · Domain normaliser ────────────────────────────────────────────────
def normalise_domain(website: str) -> str | None:
    """
    Strip protocol, www, trailing slashes and paths.
    Returns bare hostname (e.g. 'unicef.org') or None if unusable.
    """
    url = website.strip()
    if not url:
        return None

    # Prepend scheme if missing so urlparse works correctly.
    if not re.match(r"https?://", url, re.I):
        url = "https://" + url

    parsed = urlparse(url)
    host   = parsed.hostname or ""
    host   = re.sub(r"^www\.", "", host).lower().strip(".")

    return host if host else None


# Apply and preview.
df_missing["domain"] = df_missing["website"].apply(normalise_domain)

n_no_domain = df_missing["domain"].isna().sum()
print(f"records with usable domain : {len(df_missing) - n_no_domain}")
print(f"records with NO domain     : {n_no_domain}")
print()
df_missing[["org_short_name", "website", "domain"]]

In [ ]:
# ── Cell 7 · Logo fetch + validate functions ──────────────────────────────────
VALID_CONTENT_TYPES = {
    "image/png", "image/jpeg", "image/jpg",
    "image/svg+xml", "image/webp", "image/gif",
}

# Google S2 returns this generic 16-px PNG when it has nothing — skip it.
GOOGLE_FALLBACK_SIZE_BYTES = 1_114


def _make_safe_name(org_short_name) -> str:
    # str() guard handles NaN (float) or any other non-string value.
    return (
        str(org_short_name).lower()
        .replace(" ", "-")
        .replace("/", "-")
        .replace("_", "-")
        .replace("&", "")
        .replace("(", "")
        .replace(")", "")
        .replace(",", "")
    )


def _ext_from_content_type(ct: str) -> str:
    mapping = {
        "image/png":     ".png",
        "image/jpeg":    ".jpg",
        "image/jpg":     ".jpg",
        "image/svg+xml": ".svg",
        "image/webp":    ".webp",
        "image/gif":     ".gif",
    }
    return mapping.get(ct.split(";")[0].strip().lower(), ".png")


def fetch_logo(url: str, skip_if_bytes: int | None = None) -> requests.Response | None:
    """
    GET `url`. Returns the Response if it looks like a valid image, else None.
    `skip_if_bytes` rejects responses whose body is exactly that many bytes
    (used to filter Google S2's known placeholder).
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        resp.raise_for_status()
    except requests.RequestException:
        return None

    ct = resp.headers.get("Content-Type", "")
    if not any(ct.startswith(v) for v in VALID_CONTENT_TYPES):
        return None

    if skip_if_bytes is not None and len(resp.content) == skip_if_bytes:
        return None   # placeholder detected

    return resp


def fetch_logo_for_domain(domain: str) -> tuple[requests.Response, str] | tuple[None, None]:
    """
    Try source A (Clearbit) then source B (Google S2).
    Returns (response, source_label) or (None, None).
    """
    # Source A — Clearbit
    url_a = CLEARBIT_TMPL.format(domain=domain)
    resp  = fetch_logo(url_a)
    if resp:
        return resp, "clearbit"

    # Source B — Google S2
    url_b = GOOGLE_S2_TMPL.format(domain=domain)
    resp  = fetch_logo(url_b, skip_if_bytes=GOOGLE_FALLBACK_SIZE_BYTES)
    if resp:
        return resp, "google_s2"

    return None, None


print("functions defined")

In [ ]:
# ── Cell 8 · Download loop (local storage only — no Airtable writes yet) ──────
#
# Results are accumulated in `results`, a list of dicts with keys:
#   record_id, org_short_name, status, local_path, source
#
# Statuses:
#   missing_website  — no website / no parseable domain
#   logo_not_found   — both sources returned nothing usable
#   failed_request   — unexpected exception
#   downloaded       — saved to LOGOS_DIR
#   already_local    — file already exists locally, skipped re-download

def _coerce_str(val) -> str:
    """Return val as a string, treating NaN/None/empty as empty string."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return ""
    return str(val).strip()


results: list[dict] = []

for _, row in df_missing.iterrows():
    rec_id     = row["_record_id"]
    short_name = _coerce_str(row.get("org_short_name"))
    full_name  = _coerce_str(row.get("org_full_name"))
    org_name   = short_name or full_name or rec_id
    domain     = row.get("domain")

    base_result = {"record_id": rec_id, "org_short_name": org_name, "source": None, "local_path": None}

    if not domain:
        results.append({**base_result, "status": "missing_website"})
        print(f"  ⊘  {org_name:<30}  missing website")
        continue

    safe_name = _make_safe_name(org_name)

    # Check if any variant of the file already exists locally.
    existing = list(LOGOS_DIR.glob(f"{safe_name}.*"))
    if existing:
        local_path = existing[0]
        results.append({**base_result, "status": "already_local", "local_path": str(local_path)})
        print(f"  ↩  {org_name:<30}  already local → {local_path.name}")
        continue

    try:
        resp, source = fetch_logo_for_domain(domain)
    except Exception as exc:
        results.append({**base_result, "status": "failed_request", "source": str(exc)})
        print(f"  ✗  {org_name:<30}  exception: {exc}")
        continue

    if resp is None:
        results.append({**base_result, "status": "logo_not_found"})
        print(f"  ✗  {org_name:<30}  no logo found (domain: {domain})")
        continue

    ct  = resp.headers.get("Content-Type", "")
    ext = _ext_from_content_type(ct)
    filename   = f"{safe_name}{ext}"
    local_path = LOGOS_DIR / filename

    local_path.write_bytes(resp.content)

    results.append({**base_result, "status": "downloaded", "local_path": str(local_path), "source": source})
    print(f"  ✓  {org_name:<30}  {source:<12}  → {filename}")

    time.sleep(0.3)   # be polite to external APIs

print("\nDownload loop complete.")

In [ ]:
# ── Cell 9 · Results summary ──────────────────────────────────────────────────
df_results = pd.DataFrame(results)

summary = df_results.groupby("status").size().rename("count")
print("=== outcome summary ===")
print(summary.to_string())
print()

# Show records flagged for manual review.
need_review = df_results[df_results["status"].isin(["logo_not_found", "missing_website", "failed_request"])]
if not need_review.empty:
    print("--- needs manual review ---")
    print(need_review[["org_short_name", "status"]].to_string(index=False))
else:
    print("No records need manual review.")

print()
print("Full results:")
df_results

In [ ]:
# ── Cell 9.2 · Download logos that already exist in Airtable ─────────────────
#
# Cell 5 filtered OUT records that already had org_logo_white filled in Airtable,
# so those logos were never downloaded. This cell downloads them into public/logos/
# using the same slug naming convention, so Cell 9.6 can convert them to white.

import re as _re_9

def _parse_airtable_url(value: str) -> str | None:
    """Extract URL from Airtable attachment string: 'filename.ext (https://url)'"""
    if not isinstance(value, str) or not value.strip():
        return None
    m = _re_9.search(r'\(([^)]+)\)\s*$', value.strip())
    return m.group(1) if m else None


df_has_logo = df[df["org_logo_white"].str.strip() != ""].copy().reset_index(drop=True)
print(f"Records with existing Airtable logo: {len(df_has_logo)}\n")

airtable_dl_log: list[dict] = []

for _, row in df_has_logo.iterrows():
    short_name = _coerce_str(row.get("org_short_name"))
    full_name  = _coerce_str(row.get("org_full_name"))
    org_name   = short_name or full_name
    logo_value = _coerce_str(row.get("org_logo_white"))

    if not org_name:
        continue

    safe_name = _make_safe_name(org_name)

    # Skip if already present locally (any extension).
    existing = list(LOGOS_DIR.glob(f"{safe_name}.*"))
    if existing:
        airtable_dl_log.append({"org_short_name": org_name, "status": "already_local", "file": existing[0].name})
        print(f"  ↩  {org_name:<30}  already local → {existing[0].name}")
        continue

    url = _parse_airtable_url(logo_value)
    if not url:
        airtable_dl_log.append({"org_short_name": org_name, "status": "no_url"})
        print(f"  ⊘  {org_name:<30}  could not parse URL")
        continue

    try:
        resp = fetch_logo(url)
        if resp is None:
            airtable_dl_log.append({"org_short_name": org_name, "status": "download_failed"})
            print(f"  ✗  {org_name:<30}  download failed")
            continue

        ct  = resp.headers.get("Content-Type", "")
        ext = _ext_from_content_type(ct)
        filename   = f"{safe_name}{ext}"
        local_path = LOGOS_DIR / filename
        local_path.write_bytes(resp.content)
        airtable_dl_log.append({"org_short_name": org_name, "status": "downloaded", "file": filename})
        print(f"  ✓  {org_name:<30}  → {filename}")
    except Exception as exc:
        airtable_dl_log.append({"org_short_name": org_name, "status": "error", "error": str(exc)})
        print(f"  ✗  {org_name:<30}  exception: {exc}")

    time.sleep(0.2)

df_airtable_logos = pd.DataFrame(airtable_dl_log)
if not df_airtable_logos.empty:
    print()
    print("=== Airtable logo download summary ===")
    print(df_airtable_logos.groupby("status").size().to_string())

Records with existing Airtable logo: 51

  ✓  UPPSALA UNIVERSITY              → uppsala-university.png
  ✓  GiE                             → gie.png
  ✓  UNOPS                           → unops.png
  ✓  OCHA                            → ocha.png
  ✓  NRCS                            → nrcs.png
  ✓  ICG                             → icg.png
  ✓  UC IRVINE                       → uc-irvine.png
  ✓  UNESCO                          → unesco.png
  ✓  UN HABITAT                      → un-habitat.png
  ✓  WCLAC                           → wclac.png
  ✓  UNHCR                           → unhcr.png
  ✓  NWC                             → nwc.png
  ✓  Korbel                          → korbel.png
  ✓  UNU                             → unu.png
  ✓  PRIO                            → prio.png
  ✓  UNICEF                          → unicef.png
  ✓  UNDRR                           → undrr.png
  ✓  IOM                             → iom.png
  ✓  AISHA                           → aisha.png
  ✓  ICPAC      

In [ ]:
# ── Cell 9.1 · Install missing dependencies ──────────────────────────────────
# Run this once if Pillow or numpy are not yet installed in your kernel.
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "pillow", "numpy"])

In [27]:
# ── Cell 9.5 · White-conversion functions + convert df_results logos ─────────
from PIL import Image
import numpy as np

WHITE_LOGOS_DIR = Path("public") / "white_logos"
WHITE_LOGOS_DIR.mkdir(parents=True, exist_ok=True)


def is_already_white(img: Image.Image, threshold: float = 0.9) -> bool:
    """Return True if the non-transparent pixels are already majority white."""
    arr = np.array(img.convert("RGBA"), dtype=np.float32)
    opaque = arr[:, :, 3] > 50
    if not opaque.any():
        return False
    rgb = arr[opaque, :3]
    bright = (rgb.min(axis=1) > 200).sum()
    return (bright / len(rgb)) >= threshold


def convert_to_white(img: Image.Image) -> Image.Image:
    """Return a new RGBA image with the logo rendered white-on-transparent.

    For images with an alpha channel: inverted luminance × existing alpha → new alpha.
    For solid-background images (JPEG etc.): corner-sample to detect background
    brightness, then invert (light bg) or use directly (dark bg) as alpha.
    """
    img = img.convert("RGBA")
    arr = np.array(img, dtype=np.float32)
    r, g, b, a = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2], arr[:, :, 3]
    lum = 0.299 * r + 0.587 * g + 0.114 * b

    has_alpha = (a < 255).any()
    if has_alpha:
        alpha_weight = a / 255.0
        effective_dark = (1.0 - lum / 255.0) * alpha_weight
        new_alpha = (effective_dark * 255).clip(0, 255).astype(np.uint8)
    else:
        corners = [lum[0, 0], lum[0, -1], lum[-1, 0], lum[-1, -1]]
        bg_bright = np.mean(corners) > 128
        if bg_bright:
            new_alpha = (255 - lum).clip(0, 255).astype(np.uint8)
        else:
            new_alpha = lum.clip(0, 255).astype(np.uint8)

    out = np.zeros((arr.shape[0], arr.shape[1], 4), dtype=np.uint8)
    out[:, :, 0] = 255
    out[:, :, 1] = 255
    out[:, :, 2] = 255
    out[:, :, 3] = new_alpha
    return Image.fromarray(out, "RGBA")


# Convert logos from this session's download results (df_results from Cell 8).
conversion_results: list[dict] = []
to_convert = df_results[df_results["status"].isin(["downloaded", "already_local"])].copy()
to_convert = to_convert[to_convert["local_path"].notna()].copy()
print(f"Converting {len(to_convert)} logos from df_results...\n")

for _, row in to_convert.iterrows():
    org_name   = row["org_short_name"]
    local_path = Path(row["local_path"])
    record_id  = row["record_id"]
    white_path = WHITE_LOGOS_DIR / (local_path.stem + ".png")

    if white_path.exists():
        conversion_results.append({"record_id": record_id, "org_short_name": org_name,
                                    "status": "already_exists", "white_path": str(white_path)})
        print(f"  ↩  {org_name:<30}  already in white_logos/")
        continue

    if not local_path.exists():
        conversion_results.append({"record_id": record_id, "org_short_name": org_name,
                                    "status": "source_missing", "white_path": None})
        print(f"  ⊘  {org_name:<30}  source missing")
        continue

    try:
        img = Image.open(local_path)
        if is_already_white(img):
            img.convert("RGBA").save(white_path, "PNG")
            status = "copied_white"
        else:
            convert_to_white(img).save(white_path, "PNG")
            status = "converted"
        conversion_results.append({"record_id": record_id, "org_short_name": org_name,
                                    "status": status, "white_path": str(white_path)})
        print(f"  ✓  {org_name:<30}  {status}")
    except Exception as exc:
        conversion_results.append({"record_id": record_id, "org_short_name": org_name,
                                    "status": "error", "white_path": None})
        print(f"  ✗  {org_name:<30}  error: {exc}")

df_conversion = pd.DataFrame(conversion_results)
print()
print("=== conversion summary ===")
print(df_conversion.groupby("status").size().to_string())

Converting 97 logos from df_results...

  ↩  Mi Convive                      already in white_logos/
  ↩  ADHRB                           already in white_logos/
  ↩  AWSD                            already in white_logos/
  ↩  Togglecorp                      already in white_logos/
  ↩  OFCA                            already in white_logos/
  ↩  CIESIN                          already in white_logos/
  ↩  DA                              already in white_logos/
  ↩  Protect Defenders               already in white_logos/
  ↩  Mercy Corps                     already in white_logos/
  ↩  SNHR                            already in white_logos/
  ↩  SFCG                            already in white_logos/
  ↩  Ridgeway Information Ltd.,      already in white_logos/
  ↩  MVRP                            already in white_logos/
  ↩  WANEP-CI                        already in white_logos/
  ↩  ACTED                           already in white_logos/
  ↩  EOD                             already 

In [28]:
# ── Cell 9.6 · Batch-convert ALL public/logos/ → public/white_logos/ ─────────
#
# Catches any logo not covered by Cell 9.5 — including already_local files
# from previous sessions and logos downloaded via Cell 9.2.
# Idempotent: silently skips files whose white version already exists.
# Requires: WHITE_LOGOS_DIR, is_already_white, convert_to_white (Cell 9.5)

SUPPORTED_EXTS = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
all_src = sorted(f for f in LOGOS_DIR.glob("*") if f.suffix.lower() in SUPPORTED_EXTS)

n_skip = n_converted = n_copied = n_error = 0
print(f"Scanning {len(all_src)} files in {LOGOS_DIR}...\n")

for src in all_src:
    dst = WHITE_LOGOS_DIR / (src.stem + ".png")

    if dst.exists():
        n_skip += 1
        continue

    try:
        img = Image.open(src)
        if is_already_white(img):
            img.convert("RGBA").save(dst, "PNG")
            n_copied += 1
            print(f"  ↩  {src.name:<40}  already white → copied")
        else:
            convert_to_white(img).save(dst, "PNG")
            n_converted += 1
            print(f"  ✓  {src.name:<40}  converted")
    except Exception as exc:
        n_error += 1
        print(f"  ✗  {src.name:<40}  error: {exc}")

print()
print(f"  skipped (already done) : {n_skip}")
print(f"  copied (was white)     : {n_copied}")
print(f"  converted to white     : {n_converted}")
print(f"  errors                 : {n_error}")
print(f"  ─────────────────────────────────────────────")
print(f"  total in white_logos/  : {n_skip + n_copied + n_converted}")

Scanning 145 files in public/logos...

  ↩  aai.png                                   already white → copied
  ↩  acled.png                                 already white → copied
  ✓  afjc.png                                  converted
  ✓  aisha.png                                 converted
  ✓  awln.png                                  converted
  ✓  bdrcs.png                                 converted
  ↩  centre-for-humdata.png                    already white → copied
  ↩  dco.png                                   already white → copied
  ↩  dpo.png                                   already white → copied
  ↩  dppa.png                                  already white → copied
  ✓  european-commission-joint-research-centre-jrc.png  converted
  ↩  fao.png                                   already white → copied
  ✓  fhn.png                                   converted
  ↩  gie.png                                   already white → copied
  ✓  icg.png                                   con

In [1]:
df_results

NameError: name 'df_results' is not defined

In [ ]:
# ── Cell 10 · Write back to Airtable ─────────────────────────────────────────
#
# !! FINAL STEP — only run this after you are happy with the white logos !!
#
# Uses pyairtable's upload_attachment (requires pyairtable >= 2.3).
# Prefers the white-converted path from Cell 9.5; falls back to the original
# local_path if no conversion result exists for a record.
#
# Only records with a successful download AND a valid file on disk are uploaded.
# Records that already had a logo in Airtable are never touched.

# Build a record_id → white_path lookup from Cell 9.5 results.
white_path_map: dict[str, str] = {}
if "df_conversion" in dir():
    white_path_map = (
        df_conversion[df_conversion["white_path"].notna()]
        .set_index("record_id")["white_path"]
        .to_dict()
    )

# Merge: start from the download results, resolve the best file to upload.
to_upload = df_results[df_results["status"].isin(["downloaded", "already_local"])].copy()
to_upload = to_upload[to_upload["local_path"].notna()].copy()
to_upload["upload_path"] = to_upload["record_id"].map(white_path_map).fillna(to_upload["local_path"])

print(f"Records to upload : {len(to_upload)}")
print(f"Using white logo  : {to_upload['record_id'].isin(white_path_map).sum()}")
print(f"Using original    : {(~to_upload['record_id'].isin(white_path_map)).sum()}")
print()

upload_log: list[dict] = []

for _, row in to_upload.iterrows():
    rec_id      = row["record_id"]
    org_name    = row["org_short_name"]
    upload_path = Path(row["upload_path"])

    if not upload_path.exists():
        print(f"  ✗  {org_name:<30}  file not found: {upload_path}")
        upload_log.append({"org_short_name": org_name, "status": "file_missing"})
        continue

    try:
        content = upload_path.read_bytes()
        table.upload_attachment(
            record_id  = rec_id,
            field_name = LOGO_FIELD_NAME,
            filename   = upload_path.name,
            content    = content,
        )
        print(f"  ✓  {org_name:<30}  uploaded {upload_path.name}")
        upload_log.append({"org_short_name": org_name, "status": "uploaded", "file": upload_path.name})
    except Exception as exc:
        print(f"  ✗  {org_name:<30}  upload failed: {exc}")
        upload_log.append({"org_short_name": org_name, "status": "upload_failed", "error": str(exc)})

    time.sleep(0.2)   # stay within Airtable rate limits

print()
print("=== upload summary ===")
pd.DataFrame(upload_log).groupby("status").size()

In [ ]:
# ── Cell 10 · Write back to Airtable ─────────────────────────────────────────
#
# !! FINAL STEP — only run this after you are happy with the downloaded logos !!
#
# Uses pyairtable's upload_attachment (requires pyairtable >= 2.3).
# Each file is uploaded as binary content directly — no external hosting needed.
#
# Only records with status == 'downloaded' or 'already_local' are processed.
# Records that already had a logo in Airtable are never touched.

to_upload = df_results[df_results["status"].isin(["downloaded", "already_local"])].copy()
to_upload = to_upload[to_upload["local_path"].notna()]

print(f"Records to upload: {len(to_upload)}")
print()

upload_log: list[dict] = []

for _, row in to_upload.iterrows():
    rec_id     = row["record_id"]
    org_name   = row["org_short_name"]
    local_path = Path(row["local_path"])

    if not local_path.exists():
        print(f"  ✗  {org_name:<30}  file not found: {local_path}")
        upload_log.append({"org_short_name": org_name, "status": "file_missing"})
        continue

    try:
        content = local_path.read_bytes()
        table.upload_attachment(
            record_id  = rec_id,
            field_name = LOGO_FIELD_NAME,
            filename   = local_path.name,
            content    = content,
        )
        print(f"  ✓  {org_name:<30}  uploaded {local_path.name}")
        upload_log.append({"org_short_name": org_name, "status": "uploaded", "file": local_path.name})
    except Exception as exc:
        print(f"  ✗  {org_name:<30}  upload failed: {exc}")
        upload_log.append({"org_short_name": org_name, "status": "upload_failed", "error": str(exc)})

    time.sleep(0.2)   # stay within Airtable rate limits

print()
print("=== upload summary ===")
pd.DataFrame(upload_log).groupby("status").size()